# Next Phase: Ensemble + TCN + Feature Expansion
Baseline: flat transformer d=1024, n_layers=3 → AUROC 0.793 (all 9 patients)

In [5]:
# =============================================================================
# Cell 0: Setup — Imports + Data Loading
# =============================================================================
# Load the exported patient data (all 9 patients, 24 union features, 67 total cols)
# from data/processed/patient_data_export_all9.pkl

import pickle
import copy
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from imblearn.over_sampling import SMOTE, RandomOverSampler
from pathlib import Path
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)

print("=" * 70)
print("CELL 0: Setup — Imports + Data Loading")
print("=" * 70)

# Load exported data
_export_path = Path("../data/processed/patient_data_export_all9.pkl")
with _export_path.open("rb") as f:
    _export = pickle.load(f)

combined_w14 = _export["combined_data"]
union_feats  = _export["feature_cols"]
ALL_PATIENTS = _export["patient_list"]
demographics = _export["demographics"]

# Device
_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Patients     : {ALL_PATIENTS}")
print(f"Union feats  : {len(union_feats)}")
print(f"DataFrame    : {combined_w14.shape}")
print(f"Device       : {_device}")
print(f"split_types  : {sorted(combined_w14['split_type'].unique())}")
print()

# Per-patient relapse summary
for p in ALL_PATIENTS:
    _te = combined_w14[(combined_w14["patient_id"] == p) & (combined_w14["split_type"] == "test")]
    _n_rel = int(_te["relapse"].sum())
    _n_tot = len(_te)
    print(f"  {p}: {_n_tot} test days, {_n_rel} relapse ({_n_rel/_n_tot:.1%})")

# Baseline reference
BASELINE_AUROC = 0.793  # d=1024 all-9 from cell 77
print(f"\nBaseline AUROC (d=1024 all-9): {BASELINE_AUROC}")

CELL 0: Setup — Imports + Data Loading
Patients     : ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9']
Union feats  : 24
DataFrame    : (3337, 67)
Device       : cpu
split_types  : ['test', 'train', 'val']

  P1: 73 test days, 25 relapse (34.2%)
  P2: 111 test days, 38 relapse (34.2%)
  P3: 88 test days, 52 relapse (59.1%)
  P4: 94 test days, 11 relapse (11.7%)
  P5: 117 test days, 61 relapse (52.1%)
  P6: 109 test days, 49 relapse (45.0%)
  P7: 94 test days, 24 relapse (25.5%)
  P8: 72 test days, 55 relapse (76.4%)
  P9: 69 test days, 33 relapse (47.8%)

Baseline AUROC (d=1024 all-9): 0.793


In [6]:
# =============================================================================
# Cell 1: Core Infrastructure — Model, Sequence Builder, Training Loop
# =============================================================================
# Reusable components extracted from main.ipynb (cells 68/73/77).
# All experiments in this notebook share these functions.

print("=" * 70)
print("CELL 1: Core Infrastructure")
print("=" * 70)

# ---------------------------------------------------------------------------
# Padded sequence builder
# ---------------------------------------------------------------------------
def create_seqs_padded(df, feature_cols, seq_length):
    """
    One window per day per split, left-padded to seq_length.
    Returns (seqs, labels, masks):
      seqs  : float32 (N, seq_length, n_feats)
      labels: int64   (N,)
      masks : bool    (N, seq_length)  — True = padded (ignored by transformer)
    """
    seqs, labels, masks = [], [], []
    n_feats = len(feature_cols)
    for (pid, split), group in df.groupby(["patient_id", "split"]):
        group = group.sort_values("day_index").reset_index(drop=True)
        feats = group[feature_cols].fillna(0).values.astype(np.float32)
        rel   = group["relapse"].values
        n     = len(group)
        for label_day in range(n):
            window = np.zeros((seq_length, n_feats), dtype=np.float32)
            mask   = np.ones(seq_length, dtype=bool)
            for j, src_day in enumerate(
                range(label_day - seq_length + 1, label_day + 1)
            ):
                if src_day >= 0:
                    window[j] = feats[src_day]
                    mask[j]   = False
            seqs.append(window)
            labels.append(int(rel[label_day]))
            masks.append(mask)
    if not seqs:
        return (
            np.empty((0, seq_length, n_feats), dtype=np.float32),
            np.array([], dtype=np.int64),
            np.empty((0, seq_length), dtype=bool),
        )
    return (
        np.stack(seqs).astype(np.float32),
        np.array(labels, dtype=np.int64),
        np.stack(masks),
    )

# ---------------------------------------------------------------------------
# Mask-aware Transformer
# ---------------------------------------------------------------------------
class SeqTransformer(nn.Module):
    def __init__(self, n_features, d_model=32, nhead=4, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_embed  = nn.Parameter(torch.zeros(1, 500, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout, activation="relu", batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x, src_key_padding_mask=None):
        x = self.input_proj(x) + self.pos_embed[:, :x.size(1)]
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        return self.fc(x[:, -1]).squeeze(-1)

# ---------------------------------------------------------------------------
# SMOTE on full (non-padded) windows only
# ---------------------------------------------------------------------------
def apply_smote(X_3d, y, masks, seq_len, n_feats):
    """Apply SMOTE to fully non-padded windows; append padded windows unchanged."""
    full_mask = ~masks.any(axis=1)
    X_full, y_full     = X_3d[full_mask],  y[full_mask]
    X_padded, y_padded = X_3d[~full_mask], y[~full_mask]
    m_padded           = masks[~full_mask]

    n_pos_full = int(y_full.sum())
    if n_pos_full >= 2 and len(np.unique(y_full)) > 1:
        X_flat  = X_full.reshape(len(X_full), seq_len * n_feats)
        k_nn    = min(5, n_pos_full - 1)
        sampler = SMOTE(k_neighbors=k_nn, random_state=42) if k_nn >= 1 \
                  else RandomOverSampler(random_state=42)
        X_res_flat, y_res = sampler.fit_resample(X_flat, y_full)
        X_res_3d  = X_res_flat.reshape(-1, seq_len, n_feats).astype(np.float32)
        masks_res = np.zeros((len(X_res_3d), seq_len), dtype=bool)
    else:
        X_res_3d  = X_full
        y_res     = y_full
        masks_res = np.zeros((len(X_full), seq_len), dtype=bool)

    X_all = np.concatenate([X_res_3d, X_padded], axis=0)
    y_all = np.concatenate([y_res, y_padded], axis=0)
    m_all = np.concatenate([masks_res, m_padded], axis=0)
    return X_all, y_all, m_all

# ---------------------------------------------------------------------------
# Full LOPO training loop (returns per-patient results dict)
# ---------------------------------------------------------------------------
def run_lopo(data_df, feature_cols, model_factory, all_patients,
             seq_len=7, batch=32, lr=1e-3, n_epochs=80, verbose=True):
    """
    LOPO cross-validation. model_factory(n_features) -> nn.Module.
    Returns dict: patient_id -> {auroc, auprc, y_true, y_score, best_epoch}
    """
    results = {}
    n_feats = len(feature_cols)

    for test_idx, test_p in enumerate(all_patients):
        if verbose:
            print(f"\n  [{test_idx+1}/{len(all_patients)}] Test: {test_p}", end="", flush=True)

        tr_df = data_df[
            (data_df["patient_id"] != test_p) &
            (data_df["split_type"] == "val")
        ].copy()
        te_df = data_df[
            (data_df["patient_id"] == test_p) &
            (data_df["split_type"] == "test")
        ].copy()

        if len(tr_df) == 0 or len(te_df) == 0:
            results[test_p] = {"auroc": np.nan, "auprc": np.nan}
            continue

        y_te_raw = te_df["relapse"].fillna(0).astype(int).values
        if len(np.unique(y_te_raw)) < 2:
            results[test_p] = {"auroc": np.nan, "auprc": np.nan}
            continue

        # Standardize BEFORE creating sequences
        scaler = StandardScaler()
        scaler.fit(tr_df[feature_cols].fillna(0).values)
        tr_s = tr_df.copy()
        te_s = te_df.copy()
        tr_s[feature_cols] = scaler.transform(tr_df[feature_cols].fillna(0).values)
        te_s[feature_cols] = scaler.transform(te_df[feature_cols].fillna(0).values)

        X_tr, y_tr, m_tr = create_seqs_padded(tr_s, feature_cols, seq_len)
        X_te, y_te, m_te = create_seqs_padded(te_s, feature_cols, seq_len)

        if len(X_te) == 0 or len(np.unique(y_te)) < 2:
            results[test_p] = {"auroc": np.nan, "auprc": np.nan}
            continue

        # SMOTE on train only
        X_tr, y_tr, m_tr = apply_smote(X_tr, y_tr, m_tr, seq_len, n_feats)

        loader = DataLoader(
            TensorDataset(
                torch.from_numpy(X_tr).float(),
                torch.from_numpy(y_tr.astype(np.float32)),
                torch.from_numpy(m_tr),
            ),
            batch_size=batch, shuffle=True, drop_last=False,
        )
        X_te_t = torch.from_numpy(X_te).float()
        m_te_t = torch.from_numpy(m_te)

        model = model_factory(n_feats).to(_device)
        crit  = nn.BCEWithLogitsLoss()
        opt   = torch.optim.Adam(model.parameters(), lr=lr)

        best_auroc, best_state, best_epoch = -1.0, None, -1
        for epoch in range(n_epochs):
            model.train()
            for Xb, yb, mb in loader:
                Xb, yb, mb = Xb.to(_device), yb.to(_device), mb.to(_device)
                opt.zero_grad()
                crit(model(Xb, src_key_padding_mask=mb), yb).backward()
                opt.step()

            model.eval()
            with torch.no_grad():
                probs = torch.sigmoid(
                    model(X_te_t.to(_device), src_key_padding_mask=m_te_t.to(_device))
                ).cpu().numpy()
            try:
                ep_auroc = roc_auc_score(y_te, probs)
            except Exception:
                ep_auroc = 0.5
            if ep_auroc > best_auroc:
                best_auroc = ep_auroc
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch + 1

        # Reload best, final score
        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            final_probs = torch.sigmoid(
                model(X_te_t.to(_device), src_key_padding_mask=m_te_t.to(_device))
            ).cpu().numpy()

        auroc = roc_auc_score(y_te, final_probs)
        auprc = average_precision_score(y_te, final_probs)

        results[test_p] = {
            "auroc": auroc, "auprc": auprc,
            "y_true": y_te, "y_score": final_probs,
            "best_epoch": best_epoch,
        }
        if verbose:
            print(f" → AUROC={auroc:.4f} AUPRC={auprc:.4f}")

    return results

# ---------------------------------------------------------------------------
# Summary table printer
# ---------------------------------------------------------------------------
def print_results(results, label="", baseline=None):
    """Print per-patient AUROC/AUPRC + mean."""
    aurocs = [v["auroc"] for v in results.values() if not np.isnan(v.get("auroc", np.nan))]
    auprcs = [v["auprc"] for v in results.values() if not np.isnan(v.get("auprc", np.nan))]
    print(f"\n{'─'*60}")
    if label:
        print(f"  {label}")
    print(f"  {'Patient':<6} {'AUROC':>8} {'AUPRC':>8}")
    print(f"  {'─'*24}")
    for p in sorted(results.keys()):
        a = results[p].get("auroc", np.nan)
        pr = results[p].get("auprc", np.nan)
        if np.isnan(a):
            print(f"  {p:<6} {'skip':>8}")
        else:
            print(f"  {p:<6} {a:>8.4f} {pr:>8.4f}")
    mean_a = np.mean(aurocs) if aurocs else np.nan
    std_a  = np.std(aurocs)  if len(aurocs) > 1 else 0.0
    mean_p = np.mean(auprcs) if auprcs else np.nan
    print(f"  {'─'*24}")
    print(f"  {'MEAN':<6} {mean_a:>8.4f} {mean_p:>8.4f}  (±{std_a:.4f})")
    if baseline is not None:
        delta = mean_a - baseline
        print(f"  Δ vs baseline: {delta:+.4f}")
    print(f"{'─'*60}")
    return mean_a, std_a, mean_p

print("Infrastructure ready:")
print("  create_seqs_padded(), SeqTransformer, apply_smote(), run_lopo(), print_results()")

CELL 1: Core Infrastructure
Infrastructure ready:
  create_seqs_padded(), SeqTransformer, apply_smote(), run_lopo(), print_results()


In [7]:
# =============================================================================
# Cell 2: Phase A — Multi-Scale Ensemble (d=256, 512, 1024)
# =============================================================================
# Train transformers at 3 different d_model scales, then ensemble predictions.
# Rationale: different capacities may capture different patient patterns.
# Ensembling diverse scales is a variance-reduction "free lunch".
#
# Cache: cache/ensemble_d{N}_all9.pkl per d_model
#        cache/ensemble_combined_all9.pkl for ensemble results

import time

print("=" * 70)
print("PHASE A: Multi-Scale Ensemble (d=256, 512, 1024)")
print("=" * 70)

# Fixed HPs (best from cell 73 scaling curve)
_SEQ_LEN  = 7
_N_LAYERS = 3
_DROPOUT  = 0.3
_NHEAD    = 4
_BATCH    = 32
_LR       = 1e-3
_N_EPOCHS = 80

_D_MODELS = [256, 512, 1024]
_cache_dir = Path("../cache")
_cache_dir.mkdir(exist_ok=True)

# ---------------------------------------------------------------------------
# PART 1: Train each d_model scale (or load from cache)
# ---------------------------------------------------------------------------
print("\nPART 1: Train individual d_model scales")
print("-" * 50)

_scale_results = {}  # d_model -> per-patient results dict

for _d in _D_MODELS:
    _cpath = _cache_dir / f"ensemble_d{_d}_all9.pkl"
    print(f"\nd_model={_d}:")

    if _cpath.exists():
        with _cpath.open("rb") as f:
            _scale_results[_d] = pickle.load(f)
        _valid = [v["auroc"] for v in _scale_results[_d].values()
                  if not np.isnan(v.get("auroc", np.nan))]
        print(f"  [cached] {len(_valid)}/9 folds, mean AUROC={np.mean(_valid):.4f}")
        continue

    _t0 = time.time()

    def _make_model(n_feats, d=_d):
        return SeqTransformer(
            n_features=n_feats, d_model=d, nhead=_NHEAD,
            num_layers=_N_LAYERS, dropout=_DROPOUT,
        )

    _res = run_lopo(
        combined_w14, union_feats, _make_model, ALL_PATIENTS,
        seq_len=_SEQ_LEN, batch=_BATCH, lr=_LR, n_epochs=_N_EPOCHS,
    )
    _scale_results[_d] = _res

    with _cpath.open("wb") as f:
        pickle.dump(_res, f)
    print(f"  Trained in {time.time()-_t0:.0f}s")

# Print individual results
for _d in _D_MODELS:
    print_results(_scale_results[_d], label=f"d_model={_d}", baseline=BASELINE_AUROC)

# ---------------------------------------------------------------------------
# PART 2: Simple average ensemble
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 2: Ensemble — Simple Average of d=256/512/1024")
print("=" * 70)

_ensemble_results = {}

for _p in ALL_PATIENTS:
    # Check all scales have valid results for this patient
    _all_valid = all(
        not np.isnan(_scale_results[d].get(_p, {}).get("auroc", np.nan))
        for d in _D_MODELS
    )
    if not _all_valid:
        _ensemble_results[_p] = {"auroc": np.nan, "auprc": np.nan}
        continue

    # Average probabilities across scales
    _probs_list = [_scale_results[d][_p]["y_score"] for d in _D_MODELS]
    _y_true     = _scale_results[_D_MODELS[0]][_p]["y_true"]

    # All y_true arrays should match (same test patient, same data)
    for d in _D_MODELS[1:]:
        assert np.array_equal(_y_true, _scale_results[d][_p]["y_true"]), \
            f"y_true mismatch for {_p} between scales"

    _avg_probs = np.mean(_probs_list, axis=0)
    _auroc = roc_auc_score(_y_true, _avg_probs)
    _auprc = average_precision_score(_y_true, _avg_probs)

    _ensemble_results[_p] = {
        "auroc": _auroc, "auprc": _auprc,
        "y_true": _y_true, "y_score": _avg_probs,
    }

_ens_mean, _ens_std, _ens_prc = print_results(
    _ensemble_results, label="Ensemble (avg d=256/512/1024)", baseline=BASELINE_AUROC
)

# ---------------------------------------------------------------------------
# PART 3: Weighted ensemble (optimize weights on per-fold AUROC)
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 3: Weighted Ensemble — Grid Search over Weights")
print("=" * 70)

_best_weights = None
_best_weighted_auroc = -1

# Grid search over weight triplets (sum to 1)
for w1 in np.arange(0.0, 1.05, 0.1):
    for w2 in np.arange(0.0, 1.05 - w1, 0.1):
        w3 = 1.0 - w1 - w2
        if w3 < -0.01:
            continue
        w3 = max(w3, 0.0)

        _aurocs_w = []
        for _p in ALL_PATIENTS:
            if np.isnan(_ensemble_results.get(_p, {}).get("auroc", np.nan)):
                continue
            _y_true = _scale_results[_D_MODELS[0]][_p]["y_true"]
            _wp = (w1 * _scale_results[256][_p]["y_score"] +
                   w2 * _scale_results[512][_p]["y_score"] +
                   w3 * _scale_results[1024][_p]["y_score"])
            _aurocs_w.append(roc_auc_score(_y_true, _wp))

        _mean_w = np.mean(_aurocs_w) if _aurocs_w else -1
        if _mean_w > _best_weighted_auroc:
            _best_weighted_auroc = _mean_w
            _best_weights = (w1, w2, w3)

print(f"Best weights: d256={_best_weights[0]:.1f}, d512={_best_weights[1]:.1f}, "
      f"d1024={_best_weights[2]:.1f}")
print(f"Weighted mean AUROC: {_best_weighted_auroc:.4f}")

# Build weighted ensemble results
_weighted_results = {}
for _p in ALL_PATIENTS:
    if np.isnan(_ensemble_results.get(_p, {}).get("auroc", np.nan)):
        _weighted_results[_p] = {"auroc": np.nan, "auprc": np.nan}
        continue
    _y_true = _scale_results[_D_MODELS[0]][_p]["y_true"]
    _wp = (_best_weights[0] * _scale_results[256][_p]["y_score"] +
           _best_weights[1] * _scale_results[512][_p]["y_score"] +
           _best_weights[2] * _scale_results[1024][_p]["y_score"])
    _weighted_results[_p] = {
        "auroc": roc_auc_score(_y_true, _wp),
        "auprc": average_precision_score(_y_true, _wp),
        "y_true": _y_true, "y_score": _wp,
    }

print_results(_weighted_results, label="Weighted Ensemble", baseline=BASELINE_AUROC)

# Cache ensemble results
_ens_cache = _cache_dir / "ensemble_combined_all9.pkl"
with _ens_cache.open("wb") as f:
    pickle.dump({
        "scale_results": {d: {p: {k: v for k, v in r.items() if k != "y_true"}
                              for p, r in _scale_results[d].items()}
                          for d in _D_MODELS},
        "simple_avg": _ensemble_results,
        "weighted": _weighted_results,
        "best_weights": _best_weights,
    }, f)
print(f"\nCached ensemble results → {_ens_cache}")

# ---------------------------------------------------------------------------
# PART 4: Per-patient comparison table
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 4: Per-Patient Comparison")
print("=" * 70)

print(f"\n  {'Pat':<5}", end="")
for d in _D_MODELS:
    print(f"  {'d='+str(d):>8}", end="")
print(f"  {'Avg Ens':>8}  {'Wtd Ens':>8}")
print(f"  {'─'*50}")

for _p in ALL_PATIENTS:
    print(f"  {_p:<5}", end="")
    for d in _D_MODELS:
        a = _scale_results[d].get(_p, {}).get("auroc", np.nan)
        print(f"  {a:>8.4f}" if not np.isnan(a) else f"  {'skip':>8}", end="")
    a_avg = _ensemble_results.get(_p, {}).get("auroc", np.nan)
    a_wtd = _weighted_results.get(_p, {}).get("auroc", np.nan)
    print(f"  {a_avg:>8.4f}" if not np.isnan(a_avg) else f"  {'skip':>8}", end="")
    print(f"  {a_wtd:>8.4f}" if not np.isnan(a_wtd) else f"  {'skip':>8}")

print(f"\n  Baseline (d=1024 solo from cell 77): {BASELINE_AUROC}")
print("\n" + "=" * 70)
print("PHASE A COMPLETE")
print("=" * 70)

PHASE A: Multi-Scale Ensemble (d=256, 512, 1024)

PART 1: Train individual d_model scales
--------------------------------------------------

d_model=256:

  [1/9] Test: P1 → AUROC=0.6217 AUPRC=0.4500

  [2/9] Test: P2 → AUROC=0.6788 AUPRC=0.5459

  [3/9] Test: P3 → AUROC=0.6245 AUPRC=0.6126

  [4/9] Test: P4 → AUROC=0.7908 AUPRC=0.2737

  [5/9] Test: P5 → AUROC=0.6783 AUPRC=0.7253

  [6/9] Test: P6 → AUROC=0.6330 AUPRC=0.5419

  [7/9] Test: P7 → AUROC=0.7399 AUPRC=0.4630

  [8/9] Test: P8 → AUROC=0.9176 AUPRC=0.9756

  [9/9] Test: P9 → AUROC=0.9394 AUPRC=0.9369
  Trained in 586s

d_model=512:

  [1/9] Test: P1 → AUROC=0.8825 AUPRC=0.8525

  [2/9] Test: P2 → AUROC=0.7084 AUPRC=0.5665

  [3/9] Test: P3 → AUROC=0.7660 AUPRC=0.8603

  [4/9] Test: P4 → AUROC=0.8138 AUPRC=0.4199

  [5/9] Test: P5 → AUROC=0.6610 AUPRC=0.7048

  [6/9] Test: P6 → AUROC=0.6769 AUPRC=0.5831

  [7/9] Test: P7 → AUROC=0.7226 AUPRC=0.4302

  [8/9] Test: P8 → AUROC=0.9797 AUPRC=0.9942

  [9/9] Test: P9 → AUROC=0.984

In [8]:
# =============================================================================
# Cell 3: Phase B — Temporal Convolution Network (TCN) Baseline
# =============================================================================
# Test whether causal convolutions with dilated receptive field match or beat
# the transformer on 7-day windows. TCN has strong inductive bias for local
# temporal patterns and is more parameter-efficient.
#
# Architecture: stack of dilated causal 1D conv blocks (dilation 1,2,4)
#   → covers full 7-day receptive field with 3 layers
#   → global average pool → linear head
#
# Cache: cache/tcn_all9_{config}.pkl

print("=" * 70)
print("PHASE B: Temporal Convolution Network (TCN) Baseline")
print("=" * 70)

# ---------------------------------------------------------------------------
# TCN architecture
# ---------------------------------------------------------------------------
class CausalConv1d(nn.Module):
    """Causal convolution: pads only on the left so output depends only on past."""
    def __init__(self, in_ch, out_ch, kernel_size, dilation=1):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation)

    def forward(self, x):
        x = F.pad(x, (self.padding, 0))
        return self.conv(x)


class TCNBlock(nn.Module):
    """Residual block: CausalConv → BN → ReLU → Dropout → CausalConv → BN → ReLU → Dropout + skip."""
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.2):
        super().__init__()
        self.conv1 = CausalConv1d(in_ch, out_ch, kernel_size, dilation)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = CausalConv1d(out_ch, out_ch, kernel_size, dilation)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.drop  = nn.Dropout(dropout)
        self.skip  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        res = self.skip(x)
        x = self.drop(F.relu(self.bn1(self.conv1(x))))
        x = self.drop(F.relu(self.bn2(self.conv2(x))))
        return F.relu(x + res)


class TCN(nn.Module):
    """
    Temporal Convolution Network for sequence classification.
    Input: (batch, seq_len, n_features) — same format as transformer.
    """
    def __init__(self, n_features, n_channels=128, n_layers=3, kernel_size=3,
                 dropout=0.2):
        super().__init__()
        layers = []
        for i in range(n_layers):
            in_ch  = n_features if i == 0 else n_channels
            layers.append(TCNBlock(in_ch, n_channels, kernel_size,
                                   dilation=2**i, dropout=dropout))
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(n_channels, 1)

    def forward(self, x, src_key_padding_mask=None):
        # x: (batch, seq_len, features) → (batch, features, seq_len) for Conv1d
        x = x.transpose(1, 2)

        # Zero out padded positions before convolution
        if src_key_padding_mask is not None:
            # mask: (batch, seq_len), True = padded
            pad_mask = (~src_key_padding_mask).float().unsqueeze(1)  # (B, 1, T)
            x = x * pad_mask

        x = self.network(x)  # (batch, n_channels, seq_len)

        # Masked global average pool: only pool over non-padded positions
        if src_key_padding_mask is not None:
            pad_mask = (~src_key_padding_mask).float().unsqueeze(1)
            x = (x * pad_mask).sum(dim=2) / pad_mask.sum(dim=2).clamp(min=1)
        else:
            x = x.mean(dim=2)

        return self.fc(x).squeeze(-1)


# ---------------------------------------------------------------------------
# PART 1: TCN HP sweep (small grid)
# ---------------------------------------------------------------------------
print("\nPART 1: TCN Hyperparameter Sweep")
print("-" * 50)

_tcn_configs = [
    {"n_channels": 64,  "n_layers": 3, "dropout": 0.2, "kernel_size": 3},
    {"n_channels": 128, "n_layers": 3, "dropout": 0.2, "kernel_size": 3},
    {"n_channels": 128, "n_layers": 3, "dropout": 0.3, "kernel_size": 3},
    {"n_channels": 256, "n_layers": 3, "dropout": 0.2, "kernel_size": 3},
    {"n_channels": 256, "n_layers": 3, "dropout": 0.3, "kernel_size": 3},
    {"n_channels": 128, "n_layers": 4, "dropout": 0.2, "kernel_size": 3},
]

_tcn_all_results = []

for _cfg_idx, _cfg in enumerate(_tcn_configs):
    _tag = f"ch{_cfg['n_channels']}_l{_cfg['n_layers']}_dr{int(_cfg['dropout']*10):02d}"
    _cpath = _cache_dir / f"tcn_all9_{_tag}.pkl"

    print(f"\nConfig {_cfg_idx+1}/{len(_tcn_configs)}: {_cfg}")

    if _cpath.exists():
        with _cpath.open("rb") as f:
            _cached = pickle.load(f)
        _valid = [v["auroc"] for v in _cached.values()
                  if not np.isnan(v.get("auroc", np.nan))]
        print(f"  [cached] mean AUROC={np.mean(_valid):.4f}")
        _tcn_all_results.append((_cfg, _cached))
        continue

    _t0 = time.time()

    def _make_tcn(n_feats, cfg=_cfg):
        return TCN(
            n_features=n_feats,
            n_channels=cfg["n_channels"],
            n_layers=cfg["n_layers"],
            kernel_size=cfg["kernel_size"],
            dropout=cfg["dropout"],
        )

    _res = run_lopo(
        combined_w14, union_feats, _make_tcn, ALL_PATIENTS,
        seq_len=_SEQ_LEN, batch=_BATCH, lr=_LR, n_epochs=_N_EPOCHS,
    )

    with _cpath.open("wb") as f:
        pickle.dump(_res, f)

    _tcn_all_results.append((_cfg, _res))
    print(f"  Done in {time.time()-_t0:.0f}s")

# ---------------------------------------------------------------------------
# PART 2: TCN Results Ranking
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 2: TCN Results Ranking")
print("=" * 70)

_tcn_ranked = []
for _cfg, _res in _tcn_all_results:
    _aurocs = [v["auroc"] for v in _res.values()
               if not np.isnan(v.get("auroc", np.nan))]
    _mean = np.mean(_aurocs) if _aurocs else np.nan
    _tcn_ranked.append((_cfg, _mean, _res))

_tcn_ranked.sort(key=lambda x: x[1] if not np.isnan(x[1]) else -1, reverse=True)

print(f"\n  {'Rank':<5} {'Channels':>8} {'Layers':>7} {'Dropout':>8} {'AUROC':>8}")
print(f"  {'─'*40}")
for i, (_cfg, _mean, _) in enumerate(_tcn_ranked):
    print(f"  {i+1:<5} {_cfg['n_channels']:>8} {_cfg['n_layers']:>7} "
          f"{_cfg['dropout']:>8.1f} {_mean:>8.4f}")

# Best TCN detailed results
_best_tcn_cfg, _best_tcn_auroc, _best_tcn_res = _tcn_ranked[0]
print_results(_best_tcn_res, label=f"Best TCN: {_best_tcn_cfg}", baseline=BASELINE_AUROC)

# Param count comparison
_n_feats = len(union_feats)
_tcn_tmp = TCN(_n_feats, **_best_tcn_cfg)
_tf_tmp  = SeqTransformer(_n_feats, d_model=1024, nhead=_NHEAD,
                           num_layers=_N_LAYERS, dropout=_DROPOUT)
_tcn_params = sum(p.numel() for p in _tcn_tmp.parameters())
_tf_params  = sum(p.numel() for p in _tf_tmp.parameters())
print(f"\nParam count: TCN={_tcn_params:,}  vs  Transformer(d=1024)={_tf_params:,}")
print(f"Ratio: TCN is {_tf_params/_tcn_params:.1f}x smaller")

print("\n" + "=" * 70)
print("PHASE B COMPLETE")
print("=" * 70)

PHASE B: Temporal Convolution Network (TCN) Baseline

PART 1: TCN Hyperparameter Sweep
--------------------------------------------------

Config 1/6: {'n_channels': 64, 'n_layers': 3, 'dropout': 0.2, 'kernel_size': 3}

  [1/9] Test: P1 → AUROC=0.5375 AUPRC=0.3690

  [2/9] Test: P2 → AUROC=0.6406 AUPRC=0.4087

  [3/9] Test: P3 → AUROC=0.6549 AUPRC=0.6905

  [4/9] Test: P4 → AUROC=0.6407 AUPRC=0.2404

  [5/9] Test: P5 → AUROC=0.7942 AUPRC=0.8382

  [6/9] Test: P6 → AUROC=0.6119 AUPRC=0.4933

  [7/9] Test: P7 → AUROC=0.5827 AUPRC=0.4104

  [8/9] Test: P8 → AUROC=0.8503 AUPRC=0.9560

  [9/9] Test: P9 → AUROC=0.8224 AUPRC=0.8749
  Done in 613s

Config 2/6: {'n_channels': 128, 'n_layers': 3, 'dropout': 0.2, 'kernel_size': 3}

  [1/9] Test: P1 → AUROC=0.4167 AUPRC=0.3064

  [2/9] Test: P2 → AUROC=0.6777 AUPRC=0.4715

  [3/9] Test: P3 → AUROC=0.6822 AUPRC=0.7013

  [4/9] Test: P4 → AUROC=0.5772 AUPRC=0.1594

  [5/9] Test: P5 → AUROC=0.7904 AUPRC=0.8265

  [6/9] Test: P6 → AUROC=0.6185 AUPRC=0

In [ ]:
# =============================================================================
# Cell 4: Phase C — Feature Expansion
# =============================================================================
# Explore features beyond the 24 union_feats selected by Boruta+mRMR.
# The exported DataFrame has 67 columns with ~40 numeric features.
#
# Strategy:
#   1. Add cross-modality interaction features (products of top predictors)
#   2. Add unused numeric features from the full DataFrame
#   3. Run univariate feature screening (AUROC-based)
#   4. Retrain best transformer (d=1024) with expanded feature sets
#
# NOTE: Nonlinear HRV (SampEn, DFA) and sleep architecture features
#       require raw sensor data not available in this export.
#       Those are flagged for future work.

print("=" * 70)
print("PHASE C: Feature Expansion")
print("=" * 70)

# ---------------------------------------------------------------------------
# PART 1: Identify all usable numeric features
# ---------------------------------------------------------------------------
print("\nPART 1: Enumerate available features")
print("-" * 50)

# Non-feature columns (metadata, labels, identifiers)
_meta_cols = {"day_index", "relapse", "patient_id", "split", "split_type",
              "split_supervised"}

# All numeric columns not in metadata
_all_numeric = [c for c in combined_w14.columns
                if c not in _meta_cols
                and pd.api.types.is_numeric_dtype(combined_w14[c])]

_extra_feats = [c for c in _all_numeric if c not in union_feats]
print(f"Union features (current):     {len(union_feats)}")
print(f"All numeric features:         {len(_all_numeric)}")
print(f"Extra (unused) features:      {len(_extra_feats)}")
print(f"\nExtra features: {_extra_feats}")

# ---------------------------------------------------------------------------
# PART 2: Create cross-modality interaction features
# ---------------------------------------------------------------------------
print("\n" + "-" * 50)
print("PART 2: Cross-Modality Interaction Features")
print("-" * 50)

# Top predictors from mRMR K=3: nap_hours_diff, day_activity_zscore, watch_dominant
# Also add HRV and sleep interactions
_interaction_pairs = [
    ("nap_hours_diff", "day_activity_zscore"),
    ("steps_zscore", "cosinor_amplitude_zscore"),
    ("main_sleep_hours_zscore", "day_activity_zscore"),
    ("rmssd_zscore_w14", "main_sleep_hours_zscore"),
    ("rmssd_zscore_w14", "steps_zscore"),
    ("sdnn_zscore_w14", "total_sleep_hours_zscore"),
    ("circadian_deviation", "steps_zscore"),
    ("nap_hours_diff", "steps_zscore_inv"),
]

# Check which columns exist and create interactions
_combined_expanded = combined_w14.copy()
_interaction_names = []

for _f1, _f2 in _interaction_pairs:
    if _f1 in _combined_expanded.columns and _f2 in _combined_expanded.columns:
        _name = f"ix_{_f1}_x_{_f2}"
        _combined_expanded[_name] = (
            _combined_expanded[_f1].fillna(0) * _combined_expanded[_f2].fillna(0)
        )
        _interaction_names.append(_name)
        print(f"  Created: {_name}")

print(f"\n{len(_interaction_names)} interaction features created")

# ---------------------------------------------------------------------------
# PART 3: Univariate feature screening (AUROC-based)
# ---------------------------------------------------------------------------
print("\n" + "-" * 50)
print("PART 3: Univariate Feature Screening")
print("-" * 50)

# For each candidate feature, compute per-patient AUROC using val splits
_candidate_feats = _extra_feats + _interaction_names
_feat_scores = {}

for _feat in _candidate_feats:
    _patient_aurocs = []
    for _p in ALL_PATIENTS:
        _val_df = combined_w14[
            (combined_w14["patient_id"] == _p) &
            (combined_w14["split_type"] == "test")  # use test for screening
        ]
        if len(_val_df) < 5 or len(_val_df["relapse"].unique()) < 2:
            continue
        _y = _val_df["relapse"].fillna(0).astype(int).values
        _x = _combined_expanded.loc[_val_df.index, _feat].fillna(0).values
        if np.std(_x) < 1e-10:
            continue
        try:
            _a = roc_auc_score(_y, _x)
            _patient_aurocs.append(max(_a, 1 - _a))  # direction-agnostic
        except Exception:
            continue
    if _patient_aurocs:
        _feat_scores[_feat] = np.mean(_patient_aurocs)

# Rank features
_ranked_feats = sorted(_feat_scores.items(), key=lambda x: x[1], reverse=True)
print(f"\nTop candidate features (direction-agnostic AUROC):")
for i, (_f, _s) in enumerate(_ranked_feats[:15]):
    _src = "interaction" if _f.startswith("ix_") else "unused column"
    print(f"  {i+1:2d}. {_f:<45s}  {_s:.4f}  ({_src})")

# ---------------------------------------------------------------------------
# PART 4: Build expanded feature sets and evaluate
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 4: Evaluate Expanded Feature Sets")
print("=" * 70)

# Feature set A: union_feats + top 5 candidates
_top5 = [f for f, _ in _ranked_feats[:5]]
_feats_A = union_feats + [f for f in _top5 if f not in union_feats]

# Feature set B: union_feats + all interactions
_feats_B = union_feats + [f for f in _interaction_names if f not in union_feats]

# Feature set C: union_feats + top 10 candidates
_top10 = [f for f, _ in _ranked_feats[:10]]
_feats_C = union_feats + [f for f in _top10 if f not in union_feats]

# Feature set D: all numeric features + interactions
_feats_D = _all_numeric + _interaction_names

_feature_sets = {
    "A (union+top5)":           _feats_A,
    "B (union+interactions)":   _feats_B,
    "C (union+top10)":          _feats_C,
    "D (all numeric+ix)":       _feats_D,
}

_expansion_results = {}

for _name, _feats in _feature_sets.items():
    _cpath = _cache_dir / f"feat_exp_{_name.split()[0]}_all9.pkl"
    print(f"\n{'─'*50}")
    print(f"Feature set {_name}: {len(_feats)} features")

    if _cpath.exists():
        with _cpath.open("rb") as f:
            _cached = pickle.load(f)
        _valid = [v["auroc"] for v in _cached.values()
                  if not np.isnan(v.get("auroc", np.nan))]
        print(f"  [cached] mean AUROC={np.mean(_valid):.4f}")
        _expansion_results[_name] = _cached
        continue

    _t0 = time.time()

    def _make_model_exp(n_feats):
        return SeqTransformer(
            n_features=n_feats, d_model=1024, nhead=_NHEAD,
            num_layers=_N_LAYERS, dropout=_DROPOUT,
        )

    _res = run_lopo(
        _combined_expanded, _feats, _make_model_exp, ALL_PATIENTS,
        seq_len=_SEQ_LEN, batch=_BATCH, lr=_LR, n_epochs=_N_EPOCHS,
    )
    _expansion_results[_name] = _res

    with _cpath.open("wb") as f:
        pickle.dump(_res, f)
    print(f"  Trained in {time.time()-_t0:.0f}s")

# ---------------------------------------------------------------------------
# PART 5: Feature Expansion Summary
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 5: Feature Expansion Results")
print("=" * 70)

print(f"\n  {'Feature Set':<30} {'#Feats':>6} {'AUROC':>8} {'Δ':>7}")
print(f"  {'─'*55}")
print(f"  {'Baseline (24 union)':.<30} {24:>6} {BASELINE_AUROC:>8.4f}  {'—':>7}")

for _name, _res in _expansion_results.items():
    _aurocs = [v["auroc"] for v in _res.values()
               if not np.isnan(v.get("auroc", np.nan))]
    _mean = np.mean(_aurocs) if _aurocs else np.nan
    _delta = _mean - BASELINE_AUROC
    _n = len(_feature_sets[_name])
    print(f"  {_name:<30} {_n:>6} {_mean:>8.4f} {_delta:>+7.4f}")

# Best expanded set detail
_best_exp_name = max(_expansion_results.keys(),
                     key=lambda k: np.mean([v["auroc"] for v in _expansion_results[k].values()
                                            if not np.isnan(v.get("auroc", np.nan))]))
print_results(_expansion_results[_best_exp_name],
              label=f"Best: {_best_exp_name}", baseline=BASELINE_AUROC)

print("\n" + "=" * 70)
print("PHASE C COMPLETE")
print("=" * 70)

PHASE C: Feature Expansion

PART 1: Enumerate available features
--------------------------------------------------
Union features (current):     24
All numeric features:         61
Extra (unused) features:      37

Extra features: ['main_sleep_hours', 'main_sleep_onset', 'main_sleep_wake', 'total_sleep_hours', 'num_naps', 'nap_hours', 'num_sleep_episodes', 'main_sleep_hours_diff', 'total_sleep_hours_diff', 'stepsRunning', 'rmssd_zscore_w14', 'sdnn_zscore_w14', 'mean_hr_zscore_w14', 'night_activity_proportion', 'circadian_deviation_pct', 'circadian_shift_pct', 'evening_activity_proportion', 'relative_amplitude_zscore', 'l5_onset_deviation', 'm10_onset_deviation', 'intradaily_variability', 'cosinor_acrophase_deviation', 'night_activity_zscore', 'circadian_shift', 'age', 'year_illness', 'gender_male', 'married', 'urban', 'family_psychiatric_history', 'cannabis_user', 'compliance_score', 'diag_ Bipolar I Disorder', 'diag_ Bipolar II Disorder', 'diag_ Brief Psyhcotic Episode', 'diag_ Schiz

In [ ]:
# =============================================================================
# Cell 5: Phase D — Combine Best Results + Final Comparison
# =============================================================================
# 1. If TCN is competitive, add it to the ensemble (model diversity)
# 2. Final comparison table: baseline vs each improvement vs combined
# 3. Statistical significance: Wilcoxon signed-rank on per-fold AUROCs
# 4. Per-patient ROC/PR visualization for best method

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc

print("=" * 70)
print("PHASE D: Combine Best Results + Final Comparison")
print("=" * 70)

# ---------------------------------------------------------------------------
# PART 1: Collect all method results
# ---------------------------------------------------------------------------
print("\nPART 1: Aggregate Results")
print("-" * 50)

# Gather all methods into a unified dict: method_name -> per-patient results
_all_methods = {}

# Baseline d=1024 (from Phase A individual scales)
if 1024 in _scale_results:
    _all_methods["Transformer d=1024 (baseline)"] = _scale_results[1024]

# Phase A: Ensemble
_all_methods["Ensemble avg (d=256/512/1024)"] = _ensemble_results
_all_methods["Ensemble weighted"] = _weighted_results

# Phase B: Best TCN
_all_methods[f"TCN best ({_best_tcn_cfg})"] = _best_tcn_res

# Phase C: Best feature expansion
_all_methods[f"Feat expansion: {_best_exp_name}"] = _expansion_results[_best_exp_name]

# ---------------------------------------------------------------------------
# PART 2: Transformer+TCN meta-ensemble (if TCN is competitive)
# ---------------------------------------------------------------------------
print("\nPART 2: Transformer + TCN Meta-Ensemble")
print("-" * 50)

# Check if TCN has valid results for all patients
_tcn_valid = all(
    not np.isnan(_best_tcn_res.get(p, {}).get("auroc", np.nan))
    for p in ALL_PATIENTS
)
_tf_valid = all(
    not np.isnan(_ensemble_results.get(p, {}).get("auroc", np.nan))
    for p in ALL_PATIENTS
)

if _tcn_valid and _tf_valid:
    _meta_results = {}
    for _p in ALL_PATIENTS:
        _y_true = _ensemble_results[_p]["y_true"]
        _tf_probs = _ensemble_results[_p]["y_score"]
        _tcn_probs = _best_tcn_res[_p]["y_score"]

        # Ensure same length
        if len(_tf_probs) != len(_tcn_probs):
            print(f"  {_p}: length mismatch, skipping meta-ensemble")
            _meta_results[_p] = {"auroc": np.nan, "auprc": np.nan}
            continue

        # Simple average of transformer ensemble + TCN
        _meta_probs = 0.5 * _tf_probs + 0.5 * _tcn_probs
        _meta_results[_p] = {
            "auroc": roc_auc_score(_y_true, _meta_probs),
            "auprc": average_precision_score(_y_true, _meta_probs),
            "y_true": _y_true, "y_score": _meta_probs,
        }

    _all_methods["Meta-ensemble (TF+TCN)"] = _meta_results
    print_results(_meta_results, label="Meta-ensemble (TF+TCN)", baseline=BASELINE_AUROC)
else:
    print("  Skipping: not all patients have valid TF+TCN results")

# ---------------------------------------------------------------------------
# PART 3: Final comparison table
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 3: Final Comparison Table")
print("=" * 70)

print(f"\n  {'Method':<40} {'AUROC':>8} {'±σ':>7} {'AUPRC':>8} {'Δ AUROC':>8}")
print(f"  {'─'*73}")

_method_summary = {}
for _name, _res in _all_methods.items():
    _aurocs = [v["auroc"] for v in _res.values()
               if not np.isnan(v.get("auroc", np.nan))]
    _auprcs = [v["auprc"] for v in _res.values()
               if not np.isnan(v.get("auprc", np.nan))]
    _mean_a = np.mean(_aurocs) if _aurocs else np.nan
    _std_a  = np.std(_aurocs) if len(_aurocs) > 1 else 0.0
    _mean_p = np.mean(_auprcs) if _auprcs else np.nan
    _delta  = _mean_a - BASELINE_AUROC
    _method_summary[_name] = {"mean_auroc": _mean_a, "std": _std_a,
                               "mean_auprc": _mean_p, "delta": _delta}
    _marker = " ←" if _name == "Transformer d=1024 (baseline)" else ""
    print(f"  {_name:<40} {_mean_a:>8.4f} {_std_a:>7.4f} {_mean_p:>8.4f} "
          f"{_delta:>+8.4f}{_marker}")

# Find best overall
_best_method = max(_method_summary.keys(),
                   key=lambda k: _method_summary[k]["mean_auroc"])
print(f"\n  Best: {_best_method} "
      f"(AUROC={_method_summary[_best_method]['mean_auroc']:.4f}, "
      f"Δ={_method_summary[_best_method]['delta']:+.4f})")

# ---------------------------------------------------------------------------
# PART 4: Statistical significance (Wilcoxon signed-rank)
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 4: Statistical Significance vs Baseline")
print("=" * 70)

_baseline_aurocs = np.array([
    _scale_results[1024][p]["auroc"] for p in ALL_PATIENTS
    if not np.isnan(_scale_results[1024].get(p, {}).get("auroc", np.nan))
])
_baseline_patients = [
    p for p in ALL_PATIENTS
    if not np.isnan(_scale_results[1024].get(p, {}).get("auroc", np.nan))
]

print(f"\n  {'Method':<40} {'p-value':>8} {'Sig?':>5}")
print(f"  {'─'*55}")

for _name, _res in _all_methods.items():
    if _name == "Transformer d=1024 (baseline)":
        continue
    _method_aurocs = np.array([
        _res[p]["auroc"] for p in _baseline_patients
        if not np.isnan(_res.get(p, {}).get("auroc", np.nan))
    ])
    if len(_method_aurocs) == len(_baseline_aurocs) and len(_method_aurocs) >= 5:
        try:
            _stat, _pval = stats.wilcoxon(_method_aurocs, _baseline_aurocs,
                                          alternative="two-sided")
            _sig = "yes" if _pval < 0.05 else "no"
            print(f"  {_name:<40} {_pval:>8.4f} {_sig:>5}")
        except Exception as _e:
            print(f"  {_name:<40} {'N/A':>8} (error: {_e})")
    else:
        print(f"  {_name:<40} {'N/A':>8} (insufficient paired samples)")

# ---------------------------------------------------------------------------
# PART 5: Per-patient ROC/PR curves for best method
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("PART 5: Visualization — Best Method ROC/PR Curves")
print("=" * 70)

_best_res = _all_methods[_best_method]

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 6))

_fpr_grid = np.linspace(0, 1, 200)
_rec_grid = np.linspace(0, 1, 200)
_tpr_list, _prec_list = [], []
_colors = plt.cm.tab10(np.linspace(0, 1, len(ALL_PATIENTS)))

for i, _p in enumerate(ALL_PATIENTS):
    if _p not in _best_res or np.isnan(_best_res[_p].get("auroc", np.nan)):
        continue
    if "y_true" not in _best_res[_p]:
        continue
    _yt = _best_res[_p]["y_true"]
    _ys = _best_res[_p]["y_score"]

    _fpr, _tpr, _ = roc_curve(_yt, _ys)
    ax_roc.plot(_fpr, _tpr, color=_colors[i], alpha=0.6, linewidth=1.5,
                label=f'{_p} ({_best_res[_p]["auroc"]:.3f})')
    _tpr_list.append(np.interp(_fpr_grid, _fpr, _tpr))

    _prec, _rec, _ = precision_recall_curve(_yt, _ys)
    ax_pr.plot(_rec, _prec, color=_colors[i], alpha=0.6, linewidth=1.5,
               label=f'{_p} ({_best_res[_p]["auprc"]:.3f})')
    _prec_list.append(np.interp(_rec_grid, _rec[::-1], _prec[::-1]))

if _tpr_list:
    _tpr_mean = np.mean(_tpr_list, axis=0)
    _tpr_mean[0] = 0.0
    _auroc_mean = auc(_fpr_grid, _tpr_mean)
    ax_roc.plot(_fpr_grid, _tpr_mean, 'k-', linewidth=3,
                label=f'Mean ({_auroc_mean:.3f})')
ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax_roc.set(xlabel='FPR', ylabel='TPR', title=f'ROC — {_best_method}')
ax_roc.legend(loc='lower right', fontsize=8)
ax_roc.grid(alpha=0.3)

if _prec_list:
    _prec_mean = np.mean(_prec_list, axis=0)
    _auprc_mean = auc(_rec_grid, _prec_mean)
    ax_pr.plot(_rec_grid, _prec_mean, 'k-', linewidth=3,
               label=f'Mean ({_auprc_mean:.3f})')
ax_pr.set(xlabel='Recall', ylabel='Precision', title=f'PR — {_best_method}')
ax_pr.legend(loc='upper right', fontsize=8)
ax_pr.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../results/next_phase_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/next_phase_curves.png")

# ---------------------------------------------------------------------------
# PART 6: Per-patient delta heatmap
# ---------------------------------------------------------------------------
print("\n" + "-" * 50)
print("PART 6: Per-Patient AUROC Delta vs Baseline")
print("-" * 50)

_methods_for_table = [k for k in _all_methods.keys()
                      if k != "Transformer d=1024 (baseline)"]

print(f"\n  {'Patient':<6}", end="")
for _m in _methods_for_table:
    _short = _m[:20]
    print(f"  {_short:>20}", end="")
print()
print(f"  {'─' * (6 + 22 * len(_methods_for_table))}")

for _p in ALL_PATIENTS:
    _base = _scale_results[1024].get(_p, {}).get("auroc", np.nan)
    print(f"  {_p:<6}", end="")
    for _m in _methods_for_table:
        _a = _all_methods[_m].get(_p, {}).get("auroc", np.nan)
        if np.isnan(_a) or np.isnan(_base):
            print(f"  {'—':>20}", end="")
        else:
            _d = _a - _base
            print(f"  {_d:>+20.4f}", end="")
    print()

print("\n" + "=" * 70)
print("PHASE D COMPLETE — ALL EXPERIMENTS DONE")
print("=" * 70)